# TimeGPT + DTW neighbours - global one-call finetune (h = 6)

Upstream source: `DTW_TimeGPT_improved_v2.ipynb` in the thesis workbench.

**Input**: `DTW_DFS_DIR / 'dtw_3_dfs' / *.parquet` (per-keyword panels with 3 DTW neighbours; global panel learning across ~1,811 series).
**Output**: `timegpt_results_dir('dtw_timegpt_results/one_call_log1p_finetuned')` - `metrics_onecall_h6.csv`, per-series forecast CSVs under `forecasts/`, plus `logs/`.

**Leakage-safety knobs**: log1p variance stabilisation; future-known calendar features only (sin/cos); single finetune call.


In [ ]:
# --- Paper-repo bootstrap (replaces original Colab `drive.mount` cell) ---
import sys, os
from pathlib import Path

_REPO_SHARED = Path('..', '_shared').resolve()
if str(_REPO_SHARED) not in sys.path:
    sys.path.insert(0, str(_REPO_SHARED))

from paths import (  # noqa: E402
    ensure_env, DATA_ROOT, DTW_DFS_DIR, KEYWORDS_DIR_5, timegpt_results_dir,
)
from api_keys import get_nixtla_key  # noqa: E402

ensure_env()
os.environ['NIXTLA_API_KEY'] = get_nixtla_key()


In [ ]:
# =======================================
# TimeGPT — ONE global call over all keywords (leakage-safe)
# Improvements vs. baseline:
# - Global panel learning across 1,811 series
# - log1p variance stabilization (no per-series scaler)
# - Future-known calendar features only (sin/cos)
# - Finetune in a single call
# =======================================

!pip -q install nixtla polars pyarrow tqdm

import os, re, warnings
from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm
from nixtla import NixtlaClient

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- Config ----------------
BASE = DATA_ROOT
FREQ = "W-MON"
TEST_WEEKS = 6
MIN_EXTRA_WEEKS = 5  # need at least TEST_WEEKS + MIN_EXTRA_WEEKS total
IN_DIR = DTW_DFS_DIR / "dtw_3_dfs"

OUT_ROOT = timegpt_results_dir("dtw_timegpt_results/one_call_log1p_finetuned")
FC_DIR   = OUT_ROOT / "forecasts"
LOG_DIR  = OUT_ROOT / "logs"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
FC_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

METRICS_CSV = OUT_ROOT / "metrics_onecall_h6.csv"
SKIPPED_CSV = OUT_ROOT / "skipped.csv"

FINETUNE_STEPS = 20
FINETUNE_DEPTH = 2  # ignored by some client builds; harmless

# ---------------- Auth ----------------
# NIXTLA_API_KEY is set by the bootstrap cell via get_nixtla_key().

assert os.environ.get("NIXTLA_API_KEY"), "NIXTLA_API_KEY not found in environment or Colab secrets."
client = NixtlaClient(api_key=os.environ["NIXTLA_API_KEY"])

# ---------------- Helpers ----------------
import pandas as pd
def iso_to_date_robust(val):
    if hasattr(val, "isocalendar"):
        iso = val.isocalendar()
        return date.fromisocalendar(int(iso.year), int(iso.week), 1)
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    s = str(val).strip()
    m1 = re.fullmatch(r"(?P<wk>\d{1,2})-(?P<yr>\d{4})", s)  # WW-YYYY
    if m1:
        wk, yr = int(m1["wk"]), int(m1["yr"])
        if 1 <= wk <= 53:
            return date.fromisocalendar(yr, wk, 1)
    m2 = re.fullmatch(r"(?P<yr>\d{4})-(?P<wk>\d{1,2})", s)  # YYYY-WW
    if m2:
        yr, wk = int(m2["yr"]), int(m2["wk"])
        if 1 <= wk <= 53:
            return date.fromisocalendar(yr, wk, 1)
    ts = pd.to_datetime(s, errors="coerce")
    if pd.notna(ts):
        iso = ts.isocalendar()
        return date.fromisocalendar(int(iso.year), int(iso.week), 1)
    return None

def build_weekly_grid(pdf: pd.DataFrame, freq="W-MON") -> pd.DataFrame:
    if pdf.empty: return pdf
    pdf = pdf.sort_values("ds")
    num_cols = pdf.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols: return pdf[["ds"]].copy()
    pdf_num = pdf[["ds"] + num_cols].groupby("ds", as_index=False).mean()
    idx = pd.date_range(pdf_num["ds"].min(), pdf_num["ds"].max(), freq=freq)
    pdf_num = pdf_num.set_index("ds").reindex(idx)
    pdf_num.index.name = "ds"
    return pdf_num.reset_index()

def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["weekofyear"] = out["ds"].dt.isocalendar().week.astype(int)
    out["month"] = out["ds"].dt.month.astype(int)
    out["week_sin"] = np.sin(2 * np.pi * out["weekofyear"] / 52)
    out["week_cos"] = np.cos(2 * np.pi * out["weekofyear"] / 52)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    return out

def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    m = denom != 0
    return 100 * np.mean(np.abs(y_pred[m]-y_true[m]) / denom[m])

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

# ---------------- Read & assemble panel ----------------
files = sorted(IN_DIR.glob("*.parquet"))
assert files, f"No parquet files found in {IN_DIR}"

df_panel_list = []
X_future_list = []
skipped = []
te_holdout = []  # store test sets for scoring

for p in tqdm(files, desc="Prepare panel"):
    kw = p.stem
    try:
        df_pl = pl.read_parquet(p)
        if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns:
            skipped.append({"keyword": kw, "reason": "missing_cols"}); continue

        pdf0 = (
            df_pl.select(["week", "cpc_week"])
                 .with_columns(pl.col("week").map_elements(iso_to_date_robust, return_dtype=pl.Date).alias("ds"))
                 .drop("week")
                 .filter(pl.col("ds").is_not_null())
                 .sort("ds")
                 .to_pandas()
                 .rename(columns={"cpc_week":"y"})
        )
        pdf0["y"] = pd.to_numeric(pdf0["y"], errors="coerce")

        pdf = build_weekly_grid(pdf0, FREQ)
        if len(pdf) <= TEST_WEEKS + MIN_EXTRA_WEEKS:
            skipped.append({"keyword": kw, "reason": "too_short", "len": len(pdf)}); continue

        # Split
        split = len(pdf) - TEST_WEEKS
        tr = pdf.iloc[:split].copy()
        te = pdf.iloc[split:].copy()
        if te["y"].isna().any() or tr["y"].dropna().empty:
            skipped.append({"keyword": kw, "reason": "nan_in_y"}); continue

        # Simple time interpolation on train only (no leakage)
        tr = tr.set_index("ds")
        tr["y"] = tr["y"].interpolate(method="time", limit_direction="both").ffill().bfill()
        tr = tr.reset_index()

        # log1p transform (variance-stabilizing, monotone)
        tr["y_log1p"] = np.log1p(np.clip(tr["y"].values, 0, None))

        # Calendar features (bounded; do NOT scale further)
        tr_feat = add_calendar_features(tr[["ds","y_log1p"]].copy())
        CAL = ["week_sin","week_cos","month_sin","month_cos"]

        # add to panel
        tmp = tr_feat.rename(columns={"y_log1p":"y"})
        tmp["unique_id"] = kw
        df_panel_list.append(tmp[["unique_id","ds","y"] + CAL])

        # Future calendar rows for this series
        last_ds = tr["ds"].max()
        future_ds = pd.date_range(last_ds + pd.offsets.Week(weekday=0), periods=TEST_WEEKS, freq=FREQ)
        Xf = pd.DataFrame({"ds": future_ds})
        Xf = add_calendar_features(Xf)
        Xf["unique_id"] = kw
        X_future_list.append(Xf[["unique_id","ds"] + CAL])

        # Keep test for scoring
        te_holdout.append(pd.DataFrame({"unique_id": kw, "ds": te["ds"].values, "y_true": te["y"].values}))

    except Exception as e:
        skipped.append({"keyword": kw, "reason": "pipeline_error", "info": str(e)})

df_panel = pd.concat(df_panel_list, ignore_index=True)
X_df     = pd.concat(X_future_list, ignore_index=True)
df_test  = pd.concat(te_holdout, ignore_index=True)

print(f"Prepared panel: {df_panel['unique_id'].nunique()} series, {len(df_panel)} train rows")
print(f"Future rows: {len(X_df)} ; Test rows: {len(df_test)}")

# ---------------- One global call ----------------
CAL = ["week_sin","week_cos","month_sin","month_cos"]
try:
    fc = client.forecast(
        df=df_panel[["unique_id","ds","y"] + CAL],
        X_df=X_df[["unique_id","ds"] + CAL],
        h=TEST_WEEKS,
        freq=FREQ,
        level=[80],
        finetune_steps=FINETUNE_STEPS,
        finetune_depth=FINETUNE_DEPTH,
    )
except Exception:
    # Fallback: zero-shot
    fc = client.forecast(
        df=df_panel[["unique_id","ds","y"] + CAL],
        X_df=X_df[["unique_id","ds"] + CAL],
        h=TEST_WEEKS,
        freq=FREQ,
        level=[80],
    )

# ---------------- Invert transform, score, save ----------------
# Forecast column may be 'TimeGPT' (nixtla client)
pred_col = "TimeGPT" if "TimeGPT" in fc.columns else [c for c in fc.columns if c not in {"unique_id","ds"}][0]
fc = fc.rename(columns={pred_col:"yhat_log1p"})

mer = df_test.merge(fc[["unique_id","ds","yhat_log1p"]], on=["unique_id","ds"], how="inner")
mer["yhat"] = np.expm1(mer["yhat_log1p"].values)  # invert log1p

# Clip negatives due to numeric noise
mer["yhat"] = np.clip(mer["yhat"].values, 0, None)

# Metrics
per_kw = (mer.groupby("unique_id", as_index=False)
            .apply(lambda g: pd.Series({
                "smape": smape(g["y_true"], g["yhat"]),
                "rmse":  rmse(g["y_true"], g["yhat"]),
            }))
            .reset_index(drop=True))

overall = {
    "SMAPE_mean": per_kw["smape"].mean(),
    "RMSE_mean":  per_kw["rmse"].mean(),
    "N_series":   per_kw.shape[0],
}
print("Overall:", overall)

# Save outputs
per_kw.to_csv(METRICS_CSV, index=False)
pd.DataFrame(skipped).to_csv(SKIPPED_CSV, index=False)

# Save per-series forecasts
for kw, g in mer.groupby("unique_id"):
    g_out = g[["unique_id","ds","y_true","yhat"]].sort_values("ds")
    g_out.to_csv(FC_DIR / f"{kw}.csv", index=False)

print("✅ Done.")
print(f"- Metrics:   {METRICS_CSV}")
print(f"- Forecasts: {FC_DIR}")
print(f"- Skipped:   {SKIPPED_CSV}")
